# Time Series Forecasting with TorchGeo: Air Quality Tutorial

In this tutorial, we demonstrate how to train a multivariate time series forecasting model using TorchGeo. Univariate ($\mathbb{R}^{T}$) and multivariate ($\mathbb{R}^{T \times C}$) regression are the foundation of time series analysis, with several important applications in Earth science. A variety of autoregressive models can be used to tackle 1D time series, including convolution-based models like TempCNN and attention-based models like the Light Temporal Attention Encoder (L-TAE).

We train an autoregressive model on the UCI Air Quality dataset, then use it to forecast future concentrations pollutants such as Carbon Monoxide (CO), Benzene (C6$H6$), Total Nitrogen Oxides (NO$_x$), or Nitrogen Dioxide (NO$_2$).

This tutorial introduces three new TorchGeo components designed for time series:

| Component | Description |
|-----------|-------------|
| `AirQuality` / `AirQualityDataModule` | Hourly air-quality sensor data (UCI, Italian roadside station, 2004–2005) with chronological splitting and per-feature normalisation |
| `LTAE` | Lightweight Temporal Attention Encoder — encodes a variable-length sequence `(B, T, C)` into a fixed embedding `(B, d)` |
| `TemporalRegressionTask` | Lightning module that wires the encoder to a regression head, loss function, and evaluation metrics |

**What we are predicting:** given the last `NUM_PAST_STEPS` hours of readings across all sensor channels, forecast the next `NUM_FUTURE_STEPS` hours for all channels simultaneously.

## Imports

First, we import TorchGeo and any other libraries we need. `L.seed_everything` ensures reproducible weight initialisation and data shuffling across runs.

In [ ]:
import warnings
import torch
import lightning as L
from torchgeo.datasets import AirQuality
from torchgeo.datamodules import AirQualityDataModule
from torchgeo.trainers import TemporalRegressionTask
import torch.nn as nn
from torchgeo.models import LTAE
from lightning import Trainer
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
warnings.filterwarnings('ignore')
L.seed_everything(0, workers=True)

## Visualize the Dataset

**About the UCI Air Quality dataset:** The dataset contains 13 sensor channels measured every hour for approximately 13 months at a roadside station in Italy (2004–2005). Channels include concentrations of CO, non-methane hydrocarbons, benzene, total nitrogen oxides, and nitrogen dioxide, among others. We exclude the NMHC(GT) channel, which contains predominantly missing readings. Missing values encoded as `-200` in the raw data are handled automatically by `AirQuality`.

Each sample in the dataset consists of:
- **`input`**: a window of `NUM_PAST_STEPS` hourly readings across all channels (shape `T × C`)
- **`target`**: the following `NUM_FUTURE_STEPS` hourly readings (shape `H × C`)

We download the dataset and inspect its contents below.

In [ ]:
DATA_ROOT        = 'data'
NUM_PAST_STEPS   = 5
NUM_FUTURE_STEPS = 1

dataset = AirQuality(root=DATA_ROOT, download=True,
                     num_input_steps=NUM_PAST_STEPS,
                     num_target_steps=NUM_FUTURE_STEPS)

NUM_FEATURES  = len(dataset.data.columns)
dataset.data.head()


In [ ]:
sample = dataset[0]
_ = dataset.plot(sample)

## Lightning Modules

TorchGeo uses PyTorch Lightning to organise training code and set up data loaders. `AirQualityDataModule` handles loading, normalisation, and splitting in one place:

1. Downloads the data
2. Splits the dataset **chronologically** into train, validation, and test subsets
3. Applies per-feature z-score normalisation computed from the training set only

**Why chronological splitting?** Random splits would cause temporal leakage; the model would have seen future information during training, artificially inflating evaluation metrics. By splitting in time order (earliest data for training, most recent for testing), we obtain an honest estimate of generalisation performance.

The following variables can be modified to control training.

In [ ]:
dm = AirQualityDataModule(
    root=DATA_ROOT, download=True, batch_size=128, num_workers=0,
    val_split_pct=0.15, test_split_pct=0.15,
    num_input_steps=NUM_PAST_STEPS, num_target_steps=NUM_FUTURE_STEPS,
)

For training a temporal regression model, we use `TemporalRegressionTask` from `torchgeo.trainers`, which handles:

1. Encoding the input sequence through the L-TAE encoder
2. Projecting the fixed-size embedding to the output dimension via a linear head
3. Computing MSE loss in normalised space
4. Reporting RMSE and MAE in physical units (de-normalised before metric computation)

We use the **Lightweight Temporal Attention Encoder** (L-TAE; Garnot & Landrieu, 2020) as the backbone. Unlike recurrent architectures, L-TAE computes a single learned query vector that attends to *all* timesteps simultaneously, yielding a fixed-size embedding regardless of sequence length. This avoids the vanishing-gradient issues of RNNs and produces an explicit, inspectable attention weight for every timestep.

The output dimension `NUM_OUTPUTS = NUM_FUTURE_STEPS × NUM_FEATURES` reflects that we predict all channels at every future timestep simultaneously.

Key configuration choices:

1. **`n_head=4`**: Number of attention heads — each head can specialise on a different temporal pattern
2. **`d_k=8`**: Dimension of the query and key projections per head
3. **`d_model=64`**: Dimension of the encoder output embedding
4. **`n_neurons=(64, 32)`**: Hidden layer widths of the MLP regression head
5. **`dropout=0.1`**: Applied inside the attention mechanism to regularise training
6. **`loss='mse'`**: Mean squared error computed in normalised space
7. **`lr=3e-4`**: Adam learning rate

In [ ]:
task = TemporalRegressionTask(
    model='ltae', in_channels=NUM_FEATURES,
    num_outputs=NUM_FUTURE_STEPS * NUM_FEATURES,
    loss='mse', lr=3e-4, patience=5,
    n_head=4, d_k=8, d_model=64, n_neurons=(64, 32),
    dropout=0.1, len_max_seq=NUM_PAST_STEPS, T=1000,
)

## Training

We can now train the model using Lightning's `Trainer`, which incorporates:

- **`ModelCheckpoint`**: saves the weights that achieved the lowest validation loss
- **`EarlyStopping`**: halts training if validation loss does not improve for `patience` consecutive epochs, preventing overfitting

In [ ]:
trainer = Trainer(
    max_epochs=1,
    callbacks=[
        ModelCheckpoint(monitor='val_loss', mode='min', save_top_k=1),
        EarlyStopping(monitor='val_loss', patience=10, mode='min'),
    ],
)

trainer.fit(task, datamodule=dm)

## Evaluation

By using `trainer.test()`, we load the best-performing model checkpoint and perform inference on the held-out test set. Note that the data module handles automatic chronological splitting to prevent data leakage between training, validation, and test sets.

In [ ]:
trainer.test(task, datamodule=dm)

## To Go Further

- Increase `NUM_PAST_STEPS` to provide the model with a longer context window, longer histories can improve accuracy for pollutants with strong diurnal or weekly cycles
- Increase `NUM_FUTURE_STEPS` to train a multi-step-ahead forecasting model
- Replace L-TAE with a convolution-based encoder such as TempCNN for a comparison of attention-based vs. convolutional temporal modelling
- Apply the same `TemporalRegressionTask` workflow to other 1D sensor datasets, such as weather station records or streamflow gauges, by swapping in a custom dataset
- Incorporate positional encodings based on timestamps to help the model learn diurnal and seasonal patterns